# 05 · 평가 및 시각화

벤치마크 결과에서 최고 성능 모델을 자동으로 찾아 test split 평가를 수행하고,  
샘플 이미지에 예측 결과를 시각화합니다.

## ① 파라미터

In [ ]:
from pathlib import Path

# ── 수정 가능한 파라미터 ────────────────────────────────────────────
REPO_DIR       = Path(globals().get('REPO_DIR',       '/content/Military_Object_Detection'))
WORKSPACE_ROOT = Path(globals().get('WORKSPACE_ROOT', '/content/drive/MyDrive/Military_Object_Detection'))
DATASET_YAML   = Path(globals().get('DATASET_YAML',
                       str(WORKSPACE_ROOT / 'data' / 'processed' / 'yolo_dataset' / 'dataset.yaml')))

# 벤치마크 결과 루트 (latest/ 자동 탐색)
RESULTS_ROOT   = Path(globals().get('RESULTS_ROOT',
                       str(WORKSPACE_ROOT / 'experiments')))
# 직접 모델 경로 지정 시 (None이면 RESULTS_ROOT에서 자동 탐색)
MODEL_PATH     = globals().get('MODEL_PATH', None)

SPLIT    = globals().get('SPLIT',    'test')  # 'test' | 'val'
CONF     = float(globals().get('CONF',     0.25))
N_SAMPLES = int(globals().get('N_SAMPLES', 8))
NOTEBOOK_OUTPUT_DIR = Path(globals().get('NOTEBOOK_OUTPUT_DIR',
                             str(WORKSPACE_ROOT / 'artifacts' / 'notebook_eval')))
# ──────────────────────────────────────────────────────────────────

print(f'RESULTS_ROOT        : {RESULTS_ROOT}')
print(f'DATASET_YAML        : {DATASET_YAML}')
print(f'MODEL_PATH          : {MODEL_PATH or "(자동 탐색)"}')
print(f'SPLIT               : {SPLIT}')
print(f'NOTEBOOK_OUTPUT_DIR : {NOTEBOOK_OUTPUT_DIR}')

## ② 환경 초기화

In [ ]:
import sys, os
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
os.chdir(REPO_DIR)
os.environ['MAD_WORKSPACE_ROOT'] = str(WORKSPACE_ROOT)

from mad.colab_utils import setup_colab_env, check_gpu, check_dataset

setup_colab_env(REPO_DIR, WORKSPACE_ROOT)
check_gpu(require=False)
check_dataset(DATASET_YAML)

## ③ 최고 성능 모델 자동 탐색

In [ ]:
import pandas as pd
from pathlib import Path

if MODEL_PATH is not None:
    selected_model = Path(MODEL_PATH)
    print(f'직접 지정된 모델: {selected_model}')
else:
    # RESULTS_ROOT 아래에서 가장 최근 study의 benchmark_all_runs.csv 탐색
    csvs = sorted(Path(RESULTS_ROOT).rglob('benchmark_all_runs.csv'))
    if not csvs:
        raise FileNotFoundError(
            f'벤치마크 결과를 찾을 수 없습니다: {RESULTS_ROOT}\n'
            '먼저 notebooks/03_run_benchmark.ipynb를 실행하세요.'
        )
    # 가장 최근 파일 사용
    runs_csv = csvs[-1]
    print(f'결과 파일: {runs_csv}')

    df = pd.read_csv(runs_csv)
    if 'status' in df.columns:
        df = df[df['status'] == 'ok'].copy()
    if df.empty:
        raise RuntimeError('성공한 벤치마크 run이 없습니다.')

    rank_col = 'test_map50_95' if 'test_map50_95' in df.columns else 'val_map50_95'
    best_row = df.sort_values(rank_col, ascending=False).iloc[0]
    selected_model = Path(best_row['best_model'])

    print(f'\n최고 성능 모델:')
    for col in ['model_id', 'seed', 'checkpoint_type', 'val_map50_95', 'test_map50_95']:
        if col in best_row.index:
            print(f'  {col}: {best_row[col]}')

print(f'\n모델 경로: {selected_model}')
assert selected_model.exists(), f'모델 파일이 존재하지 않습니다: {selected_model}'

## ④ Test Split 평가

In [ ]:
from mad.inference import evaluate_model

eval_output_dir = NOTEBOOK_OUTPUT_DIR / 'evaluation'

metrics = evaluate_model(
    model_path=selected_model,
    dataset_yaml=DATASET_YAML,
    split=SPLIT,
    device='auto',
    output_dir=eval_output_dir,
)

print(f'\n✅ 평가 완료 (split={SPLIT})')
for k, v in metrics.items():
    if v is not None:
        print(f'  {k}: {v}')

## ⑤ 샘플 이미지 시각화

In [ ]:
import random
import yaml
import cv2
import matplotlib.pyplot as plt
from ultralytics import YOLO
from mad.utils import configure_ultralytics_env

configure_ultralytics_env()

# dataset.yaml에서 이미지 경로 로드
cfg = yaml.safe_load(DATASET_YAML.read_text(encoding='utf-8'))
dataset_root = Path(cfg.get('path', DATASET_YAML.parent))
if not dataset_root.is_absolute():
    dataset_root = (DATASET_YAML.parent / dataset_root).resolve()

split_val = cfg.get(SPLIT, cfg.get('val'))
split_path = Path(split_val)
if not split_path.is_absolute():
    split_path = (dataset_root / split_path).resolve()

if split_path.suffix.lower() == '.txt':
    image_paths = [Path(l.strip()) for l in split_path.read_text(encoding='utf-8').splitlines() if l.strip()]
else:
    VALID_EXT = {'.jpg', '.jpeg', '.png', '.bmp', '.webp', '.tif', '.tiff'}
    image_paths = [p for p in sorted(split_path.glob('*')) if p.suffix.lower() in VALID_EXT]

if not image_paths:
    raise RuntimeError(f'이미지를 찾을 수 없습니다: {split_path}')

sample_paths = random.sample(image_paths, k=min(N_SAMPLES, len(image_paths)))
model = YOLO(str(selected_model))
results = model.predict([str(p) for p in sample_paths], conf=CONF, verbose=False)

In [ ]:
ncols = 2
nrows = (len(results) + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(7 * ncols, 5 * nrows))
axes = axes.flatten() if hasattr(axes, 'flatten') else [axes]

for ax, result in zip(axes, results):
    plotted = cv2.cvtColor(result.plot(), cv2.COLOR_BGR2RGB)
    ax.imshow(plotted)
    ax.set_title(Path(result.path).name, fontsize=9)
    ax.axis('off')

for ax in axes[len(results):]:
    ax.axis('off')

plt.suptitle(f'모델: {selected_model.parent.parent.name}  |  conf={CONF}  |  split={SPLIT}', fontsize=11)
plt.tight_layout()
plt.savefig(NOTEBOOK_OUTPUT_DIR / 'sample_predictions.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'시각화 저장: {NOTEBOOK_OUTPUT_DIR}/sample_predictions.png')

## CLI 동등 명령어
```bash
# 평가
!python -m mad.inference eval \\
  --model <path/to/best.pt> \\
  --dataset-yaml $MAD_WORKSPACE_ROOT/data/processed/yolo_dataset/dataset.yaml \\
  --split test --output-dir $MAD_WORKSPACE_ROOT/artifacts/evaluation

# 추론 (이미지 폴더)
!python -m mad.inference predict \\
  --model <path/to/best.pt> \\
  --source $MAD_WORKSPACE_ROOT/data/processed/yolo_dataset/test/images \\
  --conf 0.25 --output-dir $MAD_WORKSPACE_ROOT/artifacts/inference
```